# **Part 1: Load Documents & Execute Reranking Model**

In [1]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 7.6 MB/s eta 0:00:00


In [3]:
import os
if not os.environ.get("pcsk_3zJyWq_J7iYDPPjeq4eEkwGgPcM5J1y6VCS1L6hXzkVR5saP2m6SgYrGZAH9epaW5G95Bz"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

In [4]:
from pinecone import Pinecone
api_key = os.environ.get("pcsk_3zJyWq_J7iYDPPjeq4eEkwGgPcM5J1y6VCS1L6hXzkVR5saP2m6SgYrGZAH9epaW5G95Bz")
pc = Pinecone(api_key=api_key)

In [5]:
# Define query + documents
query = "Tell me about Apple's products"
documents = [
    "Apples are fruits rich in fiber and vitamin C.",
    "Apple Inc. designs and sells iPhones, iPads, and MacBooks.",
    "A green apple can taste tart, while red apples are sweeter.",
    "Apple also builds services like iCloud and Apple Music.",
    "Granny Smith and Gala are popular varieties of apple fruit."
]

# Rerank
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3,                 # return the top 3 most relevant
    return_documents=True    # include document content back
)

# Inspect results
def show_reranked_results(query, matches):
    print(f"Query: {query}\n")
    for i, m in enumerate(matches, start=1):
        # m has .score and .document (with .id, .text if return_documents=True)
        doc_text = getattr(m.document, "text", None)
        if doc_text is None and isinstance(m.document, dict):
            doc_text = m.document.get("text", "")
        print(f"Rank {i}  |  score={m.score:.4f}\n{doc_text}\n")

show_reranked_results(query, reranked.data)  # .data holds the ranked items

Query: Tell me about Apple's products

Rank 1  |  score=0.6808
Apple Inc. designs and sells iPhones, iPads, and MacBooks.

Rank 2  |  score=0.1761
Apple also builds services like iCloud and Apple Music.

Rank 3  |  score=0.0361
Apples are fruits rich in fiber and vitamin C.



# **Part 2: Setup a Serverless Index for Medical Notes**

In [6]:
import time
import pandas as pd
from pinecone import ServerlessSpec

cloud  = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

spec = ServerlessSpec(cloud=cloud, region=region)
index_name = "medical-notes-index"

# If it exists, recreate it to start fresh
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384,       # MiniLM-L6 embedding size
    metric='cosine',
    spec=spec
)

print("✅ Index created:", index_name, "| cloud:", cloud, "| region:", region)

✅ Index created: medical-notes-index | cloud: aws | region: us-east-1


# **Part 3: Load the Sample Data**

In [7]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    raw_url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    file_path = os.path.join(tmpdir, "sample_notes_data.jsonl")

    r = requests.get(raw_url)
    r.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(r.content)

    df = pd.read_json(file_path, lines=True)

print("Data shape:", df.shape)
df.head()

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


# **Part 4: Upsert Data into the Index**

In [8]:
index = pc.Index(name=index_name)

# This helper pushes (id, vector, metadata) rows from a DataFrame in one go
index.upsert_from_dataframe(df)

def is_fresh(ix: "pc.Index"):
    stats = ix.describe_index_stats()
    vector_count = stats.total_vector_count
    print("Vector count:", vector_count)
    return vector_count > 0  # wait until at least some vectors are present

while not is_fresh(index):
    time.sleep(3)

print("✅ Index ready!")
index.describe_index_stats()

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

Vector count: 0
Vector count: 100
✅ Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

# **Part 5: Query & Embedding Function**

In [9]:
from transformers import AutoTokenizer, AutoModel
import torch

def get_embedding(input_question: str):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**encoded)
        # Average over the sequence length dimension to get a single vector
        embedding = outputs.last_hidden_state[0].mean(dim=0)
    return embedding

# Build a query and search
question = "patient with chronic knee pain and swelling"
query_vec = get_embedding(question).tolist()

results = index.query(vector=[query_vec], top_k=8, include_metadata=True)
# Sort strictly descending (high score = more similar)
matches_sorted = sorted(results['matches'], key=lambda m: m['score'], reverse=True)

# Display a few
def show_results(q, matches):
    print(f"Question: '{q}'\n")
    for i, m in enumerate(matches, start=1):
        print(f"{str(i).rjust(2)}. ID: {m['id']}")
        print(f"   Score: {m['score']}")
        print(f"   Metadata: {m['metadata']}\n")

show_results(question, matches_sorted[:5])

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Question: 'patient with chronic knee pain and swelling'

 1. ID: P007
   Score: 0.704579413
   Metadata: {'surgery': 'knee arthroscopy', 'symptoms': 'pain, swelling', 'treatment': 'physical therapy'}

 2. ID: P028
   Score: 0.623988509
   Metadata: {'condition': 'knee pain', 'referral': 'orthopedics'}

 3. ID: P059
   Score: 0.527732849
   Metadata: {'symptoms': 'joint pain', 'treatment': 'NSAIDs, rest'}

 4. ID: P0100
   Score: 0.444148332
   Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}

 5. ID: P047
   Score: 0.435629338
   Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}



# **Part 6: Display & Rerank Clinical Notes**

In [10]:
# Prepare documents for reranking: concatenate metadata into a single field
transformed_documents = [
    {
        "id": m["id"],
        "reranking_field": "; ".join([f"{k}: {v}" for k, v in m["metadata"].items()])
    }
    for m in matches_sorted
]

refined_query = "recommended treatment approach for osteoarthritis knee pain"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True
)

def show_reranked_results(q, ranked):
    print(f"Refined Question: '{q}'\nReranked Results:\n")
    for i, m in enumerate(ranked, start=1):
        # m.score is the reranking score, m.document is the doc we passed
        doc = m.document
        # Document may be a dict or an object; handle both
        rid  = getattr(doc, "id", None) or (doc.get("id") if isinstance(doc, dict) else None)
        rtxt = getattr(doc, "reranking_field", None) or (doc.get("reranking_field") if isinstance(doc, dict) else None)
        print(f"{i}. ID: {rid}")
        print(f"   Score: {m.score:.4f}")
        print(f"   Reranking Field: {rtxt}\n")

show_reranked_results(refined_query, reranked_results.data)

Refined Question: 'recommended treatment approach for osteoarthritis knee pain'
Reranked Results:

1. ID: P007
   Score: 0.1184
   Reranking Field: surgery: knee arthroscopy; symptoms: pain, swelling; treatment: physical therapy

2. ID: P059
   Score: 0.0642
   Reranking Field: symptoms: joint pain; treatment: NSAIDs, rest

3. ID: P0100
   Score: 0.0421
   Reranking Field: advice: over-the-counter pain relief, stretching; symptoms: muscle pain

